# Notebook 02 — CLIP Text→Image Search

In Notebook 01 we saw images become token sequences. Now: **can we map images and text into the *same* vector space**, so we can search a folder of photos with a sentence?

Yes. That's CLIP.

## The idea (S3 §4.1)

OpenAI's CLIP (Contrastive Language-Image Pretraining, 2021) trains two encoders **jointly** on 400M (image, caption) pairs from the web:

```
text  →  Text encoder  → 512-d vector
image →  Image encoder → 512-d vector
```

The training objective is **contrastive**: pull matching (image, caption) pairs together in vector space, push non-matching pairs apart. After 400M pairs, *"a photo of a cat"* lands near photos of cats and far from photos of cars.

What this unlocks:
- **Text→image search** — embed text query, cosine-search image embeddings
- **Zero-shot classification** — embed candidate labels, pick the closest
- **Image→image search** — find visually similar images

We'll build text→image search on a small photo gallery.

## 1. Load CLIP

We'll use `openai/clip-vit-base-patch32`. ~150 MB, runs on CPU.

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "openai/clip-vit-base-patch32"

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Device: {device}")

model = CLIPModel.from_pretrained(MODEL_NAME).to(device).eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

# Both encoders output 512-d vectors. Patch size 32 → for 224×224 input, 7×7=49 patches.
print(f"Embedding dim: {model.config.projection_dim}")

## 2. Build a small photo gallery

We'll fetch ~10 photos covering diverse subjects so search results are easy to eyeball.

In [ ]:
import io
import urllib.request
from PIL import Image

# (label, unsplash URL). Labels are only for our debugging — CLIP never sees them.
GALLERY = [
    ("sushi",         "https://images.unsplash.com/photo-1579871494447-9811cf80d66c?w=400"),
    ("ramen",         "https://images.unsplash.com/photo-1557872943-16a5ac26437e?w=400"),
    ("pizza",         "https://images.unsplash.com/photo-1513104890138-7c749659a591?w=400"),
    ("torii_gate",    "https://images.unsplash.com/photo-1528164344705-47542687000d?w=400"),
    ("snow_mountain", "https://images.unsplash.com/photo-1551524559-8af4e6624178?w=400"),
    ("beach",         "https://images.unsplash.com/photo-1507525428034-b723cf961d3e?w=400"),
    ("golden_retriever", "https://images.unsplash.com/photo-1552053831-71594a27632d?w=400"),
    ("black_cat",     "https://images.unsplash.com/photo-1548247416-ec66f4900b2e?w=400"),
    ("laptop_desk",   "https://images.unsplash.com/photo-1517336714731-489689fd1ca8?w=400"),
    ("coffee_cup",    "https://images.unsplash.com/photo-1509042239860-f550ce710b93?w=400"),
]

images, labels = [], []
for label, url in GALLERY:
    try:
        b = urllib.request.urlopen(url, timeout=10).read()
        images.append(Image.open(io.BytesIO(b)).convert("RGB"))
        labels.append(label)
    except Exception as e:
        print(f"Skipping {label}: {e}")
print(f"Loaded {len(images)} images.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, img, lab in zip(axes.flat, images, labels):
    ax.imshow(img); ax.set_title(lab, fontsize=9); ax.set_axis_off()
plt.tight_layout(); plt.show()

## 3. Embed all images once

Production note: in a real system you'd do this in batches and cache the vectors. CLIP image embedding is ~one shot per image — never re-embed.

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def embed_images(imgs):
    inputs = processor(images=imgs, return_tensors="pt").to(device)
    feats = model.get_image_features(**inputs)
    return F.normalize(feats, dim=-1)   # L2-normalize → cosine becomes dot product

@torch.no_grad()
def embed_texts(texts):
    inputs = processor(text=texts, return_tensors="pt", padding=True).to(device)
    feats = model.get_text_features(**inputs)
    return F.normalize(feats, dim=-1)

image_vectors = embed_images(images)
print("Image embedding tensor shape:", image_vectors.shape)   # (N, 512)

## 4. Search

Embed a text query, dot-product against image vectors, sort. That's the whole show.

In [ ]:
def search(query: str, k: int = 3):
    q_vec = embed_texts([query])                          # (1, 512)
    sims = (q_vec @ image_vectors.T).squeeze(0)            # (N,)
    topk = sims.topk(k)
    return [(labels[i], float(sims[i]), images[i]) for i in topk.indices.tolist()]


def show_results(query, hits):
    fig, axes = plt.subplots(1, len(hits), figsize=(4 * len(hits), 4))
    if len(hits) == 1:
        axes = [axes]
    fig.suptitle(f"Query: '{query}'", fontsize=12)
    for ax, (lab, sim, im) in zip(axes, hits):
        ax.imshow(im); ax.set_title(f"{lab}\nsim={sim:.3f}"); ax.set_axis_off()
    plt.tight_layout(); plt.show()


for q in ["sushi", "a torii gate at sunset", "snowy mountain peak", "a fluffy dog"]:
    show_results(q, search(q, k=3))

Notice that *"a torii gate at sunset"* still finds the torii gate even though no caption ever mentioned the word "sunset" — CLIP's text encoder generalizes from the 400M caption distribution. That generalization is why CLIP-style retrieval is the default for natural-image catalogs.

## 5. Where CLIP-style retrieval breaks

CLIP works beautifully on **natural images**. It's mediocre on **text-heavy documents** — PDFs, slides, screenshots. Two reasons:

1. **The modality gap (S3 §4.2):** in CLIP's vector space, text embeddings cluster in one region and image embeddings in another. Hurts retrieval quality at the margin.
2. **Single-vector compression:** a whole page → one 512-d vector. The chart bar that says "Q3 = $4.2M" gets averaged with everything else on the page.

Let's see this fail.

In [ ]:
# Ask something that needs reading text inside the image
show_results("the number 4.2 million", search("the number 4.2 million", k=3))
show_results("Q3 revenue", search("Q3 revenue", k=3))

These return **whatever's vaguely closest in image-vibes**, not what we'd expect. That's not a bug in our code — it's CLIP's design. CLIP embeds the *whole image* into one vector. Fine details (numbers in a chart, words on a sign) don't survive that compression.

**This is exactly the gap ColPali fixes.** Instead of one vector per page, ColPali keeps ~1024 patch vectors per page and uses *late interaction* to match query tokens against specific patches. We'll build that in Notebook 03.

## What we learned

- CLIP gives you text-and-image-in-the-same-vector-space for free, in 512 dimensions.
- Text→image search is one cosine multiplication — ~1 ms over 100k images on CPU with FAISS or HNSW.
- CLIP is the right tool for **natural images**: photo catalogs, e-commerce, stock libraries.
- CLIP is the **wrong** tool for text-heavy documents. The single-vector bottleneck loses the details that matter most.

**Next:** [Notebook 03 — ColPali Visual RAG](03_colpali_visual_rag.ipynb). Same idea, but with multiple vectors per page and late interaction — the architecture every modern document AI product is built on.